# CLI Workflows & Automation

qb-compiler ships with a command-line interface (`qbc`) for common compilation and
analysis tasks. This notebook demonstrates CLI usage via `subprocess` calls and shows
how to integrate `qbc` into CI/CD pipelines and automated workflows.

**Note:** This notebook runs CLI commands via `subprocess`. The `qbc` CLI must be
installed (`pip install qb-compiler[cli]` or from source).

Commands covered:
- `qbc doctor`: environment check
- `qbc info`: version and backend listing
- `qbc preflight`: quick viability check
- `qbc analyze`: detailed circuit analysis
- `qbc compile`: compilation with `--receipt`
- `qbc diff`: backend comparison
- `qbc calibration show`: calibration data inspection
- Automation patterns: batch preflight, CI/CD hooks

In [1]:
import subprocess
import json
import tempfile
from pathlib import Path


def run_qbc(*args, check=True):
    """Run a qbc CLI command and return stdout."""
    result = subprocess.run(
        ["qbc", *args],
        capture_output=True,
        text=True,
        timeout=60,
    )
    if check and result.returncode != 0:
        print(f"STDERR: {result.stderr}")
    print(result.stdout)
    return result

## 1. qbc doctor and qbc info

`qbc doctor` verifies your quantum development environment: Python version, Qiskit,
IBM credentials, calibration snapshots, and optional dependencies.
`qbc info` shows the installed version and lists all configured backends.

In [2]:
run_qbc("doctor")
print("\n" + "=" * 60 + "\n")
run_qbc("info");


qbc doctor

✔  qb-compiler 0.5.1.post1.dev1+g42eddfbf9.d20260430
✔  Python 3.11.14
✔  Qiskit 1.4.5
✔  IBM credentials configured (3 account(s))
✔  9 backends configured
✔  5 calibration snapshot(s) available
         ibm_fez_2026_02_15.json
         ibm_fez_2026_03_12.json
         ibm_fez_2026_03_14.json
         ibm_torino_2026_02_15.json
         rigetti_ankaa_2026_02_15.json
✔  ML layout predictor available
✔  QubitBoost SDK 0.2.0
✔  numpy 2.2.6
✔  rustworkx 0.17.1
✔  rich ?

Environment looks good!






qb-compiler 0.5.1.post1.dev1+g42eddfbf9.d20260430

Available backends:
  ibm_fez               ibm        156q  $0.00016/shot
  ibm_marrakesh         ibm        156q  $0.00016/shot
  ibm_torino            ibm        133q  $0.00014/shot
  ionq_aria             ionq        25q  $0.30000/shot
  ionq_forte            ionq        36q  $0.30000/shot
  iqm_emerald           iqm          5q  $0.00020/shot
  iqm_garnet            iqm         20q  $0.00045/shot
  quantinuum_h2         quantinuum    32q  $8.00000/shot
  rigetti_ankaa         rigetti     84q  $0.00035/shot



## 2. Create test circuits

The CLI commands operate on QASM files. Let's create a GHZ circuit to use throughout.

In [3]:
from qiskit import QuantumCircuit, qasm2

work_dir = Path(tempfile.mkdtemp(prefix="qbc_demo_"))

qc = QuantumCircuit(4, name="ghz_4")
qc.h(0)
for i in range(3):
    qc.cx(i, i + 1)
qc.measure_all()

qasm_path = work_dir / "ghz_4.qasm"
qasm2.dump(qc, qasm_path)

print(f"Circuit saved to: {qasm_path}")
print(f"\nQASM content:")
print(qasm_path.read_text())

Circuit saved to: /tmp/qbc_demo_v__0ty_j/ghz_4.qasm

QASM content:
OPENQASM 2.0;
include "qelib1.inc";
qreg q[4];
creg meas[4];
h q[0];
cx q[0],q[1];
cx q[1],q[2];
cx q[2],q[3];
barrier q[0],q[1],q[2],q[3];
measure q[0] -> meas[0];
measure q[1] -> meas[1];
measure q[2] -> meas[2];
measure q[3] -> meas[3];



## 3. qbc preflight: quick viability check

Returns one of three verdicts: **VIABLE**, **CAUTION**, or **DO NOT RUN**.

```
qbc preflight <circuit.qasm> --backend <backend> [--seeds N]
```

In [4]:
run_qbc("preflight", str(qasm_path), "--backend", "ibm_fez");


  Circuit: circuit-157
  Backend: ibm_fez (156q)

  Status: VIABLE
  Estimated fidelity: 0.8771
  Depth: 8  (viable limit: 406)
  2Q gates: 3
  Cost (4096 shots): $0.6554

  QubitBoost gate eligibility:
    * TomoGate       Available: Pre-flight state fidelity certification
    * LiveGate       Available: Real-time doom detection
    * ShotValidator  Available: Result integrity verification

  QubitBoost SDK installed.




## 4. qbc analyze and qbc compile

`qbc analyze` provides circuit type detection, gate breakdown, signal-to-noise ratio, and
actionable suggestions. `qbc compile` compiles with optional `--receipt` for metadata.

Strategies: `fidelity_optimal` (default), `depth_optimal`, `budget_optimal`

In [5]:
run_qbc("analyze", str(qasm_path), "--backend", "ibm_fez");


  Circuit Analysis: circuit-157
  Circuit type: General
  Qubits: 4  Gates: 9  Depth: 5
  Gate breakdown: measure:4, cx:3, h:1, barrier:1

  Backend: ibm_fez (156q)
  Status: VIABLE
  Estimated fidelity: 0.8771
  Signal/noise ratio: 14.0x
  Depth: 8  (viable limit: 406)
  2Q gates after transpilation: 3
  Cost (4096 shots): $0.6554

  Estimated fidelity 0.877 is well above noise floor (0.0625). Circuit should produce meaningful results.

  Suggestions:
    - Circuit looks good: proceed with execution.
    - Calibration snapshot is 90 days old; the estimate reflects that date. Pass fresh backend_props or set QBC_CALIBRATION_DIR for current numbers.

  QubitBoost gate eligibility:
    * TomoGate       Available: Pre-flight state fidelity certification
    * LiveGate       Available: Real-time doom detection
    * ShotValidator  Available: Result integrity verification

  QubitBoost SDK installed.




In [6]:
run_qbc(
    "compile", str(qasm_path),
    "--backend", "ibm_fez",
    "--strategy", "fidelity_optimal",
    "--receipt",
)

# Check if a receipt was generated
receipt_path = qasm_path.with_suffix(".receipt.json")
if receipt_path.exists():
    receipt = json.loads(receipt_path.read_text())
    print("Compilation receipt:")
    print(json.dumps(receipt, indent=2))
else:
    print("No receipt generated (compile may have been skipped).")

Compiled: depth 4 -> 15 (-275.0% reduction)
Estimated fidelity: 0.9922
Compilation time: 5881.7 ms
Receipt saved to /tmp/qbc_demo_v__0ty_j/ghz_4.receipt.json
View execution history and trends at https://qubitboost.io/dashboard

Compilation receipt:
{
  "qb_compiler_version": "0.5.1.post1.dev1+g42eddfbf9.d20260430",
  "timestamp": "2026-06-12T11:36:19.682847+00:00",
  "circuit_file": "/tmp/qbc_demo_v__0ty_j/ghz_4.qasm",
  "backend": "ibm_fez",
  "strategy": "fidelity_optimal",
  "original_depth": 4,
  "compiled_depth": 15,
  "depth_reduction_pct": -275.0,
  "estimated_fidelity": 0.992187,
  "compilation_time_ms": 5881.72,
  "gate_count": 24
}


## 5. qbc diff and qbc calibration show

`qbc diff` compares circuit performance on two backends side by side.
`qbc calibration show` displays the calibration summary for a backend.

In [7]:
run_qbc(
    "diff", str(qasm_path),
    "--backend", "ibm_fez",
    "--vs", "ibm_torino",
);


  Circuit: circuit-157

                                       ibm_fez         ibm_torino
                              ----------------   ----------------
  Status                                VIABLE             VIABLE
  Est. fidelity                         0.8771             0.9252 <
  2Q gates                                   3 <                3
  Depth                                      8                  5 <
  Viable depth                             406                337
  SNR                                    14.0x              14.8x
  Cost/4096 shots                      $0.6554            $0.5734 <

  Recommendation: ibm_torino (+0.0481 fidelity)

  Live calibration comparison available at https://qubitboost.io/compiler




In [8]:
for backend in ["ibm_fez", "ibm_torino", "ibm_marrakesh"]:
    print(f"=== {backend} ===")
    run_qbc("calibration", "show", backend)
    print()

=== ibm_fez ===


Backend: ibm_fez
Provider: ibm
Qubits: 156
Basis gates: id, rz, sx, x, cz, reset
Median CX error: 0.0050
Median readout error: 0.0100
Median T1: 300.0 us
Median T2: 150.0 us
Cost per shot: $0.00016


=== ibm_torino ===


Backend: ibm_torino
Provider: ibm
Qubits: 133
Basis gates: id, rz, sx, x, cz, reset
Median CX error: 0.0060
Median readout error: 0.0120
Median T1: 280.0 us
Median T2: 130.0 us
Cost per shot: $0.00014


=== ibm_marrakesh ===


Backend: ibm_marrakesh
Provider: ibm
Qubits: 156
Basis gates: id, rz, sx, x, cz, reset
Median CX error: 0.0055
Median readout error: 0.0110
Median T1: 290.0 us
Median T2: 140.0 us
Cost per shot: $0.00016




## 6. Scripting: batch preflight checks

Run viability checks across multiple circuits and backends to find the best
circuit-backend pairings before spending QPU time.

In [9]:
from qiskit import qasm2 as _qasm2

# Create additional test circuits
for name, builder in [
    ("bell", lambda: (q := QuantumCircuit(2, name="bell"), q.h(0), q.cx(0, 1), q.measure_all(), q)[-1]),
    ("ghz_8", lambda: (q := QuantumCircuit(8, name="ghz_8"), q.h(0), [q.cx(i, i+1) for i in range(7)], q.measure_all(), q)[-1]),
]:
    circ = builder()
    (work_dir / f"{name}.qasm").write_text(_qasm2.dumps(circ))

# Batch preflight
backends = ["ibm_fez", "ibm_torino"]
circuit_names = ["bell", "ghz_4", "ghz_8"]

print(f"{'Circuit':>15s} | {'Backend':>12s} | Result")
print("-" * 50)

for name in circuit_names:
    qasm_file = work_dir / f"{name}.qasm"
    for backend in backends:
        result = subprocess.run(
            ["qbc", "preflight", str(qasm_file), "--backend", backend],
            capture_output=True, text=True, timeout=60,
        )
        status = "ERROR"
        for line in result.stdout.splitlines():
            if "Status:" in line:
                status = line.split("Status:")[1].strip()
                break
        print(f"{name:>15s} | {backend:>12s} | {status}")

        Circuit |      Backend | Result
--------------------------------------------------


           bell |      ibm_fez | VIABLE


           bell |   ibm_torino | VIABLE


          ghz_4 |      ibm_fez | VIABLE


          ghz_4 |   ibm_torino | VIABLE


          ghz_8 |      ibm_fez | VIABLE


          ghz_8 |   ibm_torino | ERROR


## 7. CI/CD integration patterns

### Pre-commit hook

```bash
#!/bin/bash
# .git/hooks/pre-commit: block commits with non-viable circuits

QASM_FILES=$(git diff --cached --name-only --diff-filter=ACM | grep '\.qasm$')
[ -z "$QASM_FILES" ] && exit 0

echo "Running qbc preflight on staged QASM files..."
FAILED=0

for f in $QASM_FILES; do
    OUTPUT=$(qbc preflight "$f" --backend ibm_fez 2>&1)
    if echo "$OUTPUT" | grep -q "DO NOT RUN"; then
        echo "FAIL: $f is not viable on ibm_fez"
        FAILED=1
    fi
done

[ $FAILED -ne 0 ] && echo "Commit blocked: non-viable circuits." && exit 1
```

### GitHub Actions workflow

```yaml
# .github/workflows/quantum-ci.yml
name: Quantum Circuit CI
on:
  push:
    paths: ['**.qasm', '**.py']
jobs:
  preflight:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install qb-compiler[cli]
      - run: qbc doctor
      - name: Preflight all QASM files
        run: |
          find . -name '*.qasm' | while read f; do
            echo "Checking: $f"
            qbc preflight "$f" --backend ibm_fez || true
          done
      - name: Compile with receipts
        run: |
          find . -name '*.qasm' | while read f; do
            qbc compile "$f" --backend ibm_fez --receipt || true
          done
      - uses: actions/upload-artifact@v4
        with:
          name: compilation-receipts
          path: '**/*.receipt.json'
```

## 8. Automated compilation pipeline

Combine multiple `qbc` commands into a pipeline that processes a directory of QASM
files and generates a summary report.

In [10]:
def compilation_pipeline(circuit_dir, backend="ibm_fez"):
    """Run a full compilation pipeline on all QASM files in a directory."""
    circuit_dir = Path(circuit_dir)
    qasm_files = sorted(circuit_dir.glob("*.qasm"))

    if not qasm_files:
        print(f"No QASM files found in {circuit_dir}")
        return []

    results = []
    for qasm_file in qasm_files:
        entry = {"file": qasm_file.name, "backend": backend}

        result = subprocess.run(
            ["qbc", "preflight", str(qasm_file), "--backend", backend],
            capture_output=True, text=True, timeout=60,
        )
        for line in result.stdout.splitlines():
            if "Status:" in line:
                entry["status"] = line.split("Status:")[1].strip()
            if "Estimated fidelity:" in line:
                entry["fidelity"] = line.split(":")[1].strip()

        if entry.get("status") in ("VIABLE", "CAUTION"):
            compile_result = subprocess.run(
                ["qbc", "compile", str(qasm_file), "--backend", backend, "--receipt"],
                capture_output=True, text=True, timeout=60,
            )
            entry["compiled"] = compile_result.returncode == 0
        else:
            entry["compiled"] = False

        results.append(entry)
    return results


pipeline_results = compilation_pipeline(work_dir, backend="ibm_fez")
print("Pipeline results:")
print(json.dumps(pipeline_results, indent=2))

# Clean up
import shutil
shutil.rmtree(work_dir, ignore_errors=True)
print(f"\nCleaned up {work_dir}")

Pipeline results:
[
  {
    "file": "bell.qasm",
    "backend": "ibm_fez",
    "status": "VIABLE",
    "fidelity": "0.9430",
    "compiled": true
  },
  {
    "file": "ghz_4.qasm",
    "backend": "ibm_fez",
    "status": "VIABLE",
    "fidelity": "0.8771",
    "compiled": true
  },
  {
    "file": "ghz_8.qasm",
    "backend": "ibm_fez",
    "status": "VIABLE",
    "fidelity": "0.8604",
    "compiled": true
  }
]

Cleaned up /tmp/qbc_demo_v__0ty_j


## CLI command reference

| Command | Purpose | Key flags |
|---------|---------|----------|
| `qbc doctor` | Environment check |: |
| `qbc info` | Version + backends |: |
| `qbc preflight` | Quick viability check | `--backend`, `--seeds` |
| `qbc analyze` | Detailed analysis | `--backend`, `--seeds` |
| `qbc compile` | Compile circuit | `--backend`, `--strategy`, `--receipt`, `--output` |
| `qbc diff` | Compare two backends | `--backend`, `--vs`, `--seeds` |
| `qbc calibration show` | Calibration summary | `<backend>` |